# Lossy materials using conductivity
Currently, FDTDX supports using lossy materials through the setting of a real- and scalar-valued conductivity $\sigma$,
through the parameter named `electric_conductivity` in `Material`.
So far, it implements the vector form of Ohm's law, $\mathbf{J}_f = \sigma \mathbf{E}$, where $\mathbf{J}_f$ is the free current density.
This is then applied to the macroscopic Ampère-Maxwell equation,

$$
\mathbf{\nabla} \times \mathbf{H} = \mathbf{J}_f + \frac{\partial \mathbf{D}}{\partial t},
$$

where 

$$
{\begin{aligned}\mathbf {D} (\mathbf{r},t) &= \varepsilon_{0}\mathbf{E} (\mathbf{r} ,t) + \mathbf{P}(\mathbf{r} ,t), \\
\mathbf{H} (\mathbf{r},t) &= {\frac{1}{\mu_{0}}}\mathbf{B}(\mathbf{r},t) - \mathbf{M}(\mathbf{r}, t), \end{aligned}}
$$

Conductivity can be used either to model losses due to free currents (like those induced by an electric field on a non-ideal metal), or, if the right assumptions hold, due to bound currents originating from polarization lag (dipoles oscillating do not instantly move once an electric field impinges on them) in a lossy dielectric, making it equivalent to specifying an *imaginary* component, $\varepsilon''$, of a complex-valued dielectric permittivity $\varepsilon$. The *real* component of $\varepsilon$, which is $\varepsilon'$, is what's specified as the relative permittivity (so $\varepsilon' = \varepsilon'_r \varepsilon_0$) in the `permittivity` parameter.

FDTDX assumes that we are using linear materials ($\mathbf{P} = \chi \mathbf{E}$, $\mathbf{M} = \chi_m \mathbf{H}$, which implies that we can write $\mathbf{D} = \varepsilon \mathbf{E}$ and $\mathbf{H} = \frac{1}{\mu} \mathbf{B}$), as well as time-invariant ones ($\mu$ and $\varepsilon$ constant with time; we can take these two out of the partial derivatives with respect to time). Since this means that the Maxwell equations are also linear, contain no explicit time dependence (aside from partial derivatives) then the electric and magnetic fields can each be expressed as a product of factors, each only depending on space or time. Using these assumptions, we can furthermore express the time-dependent factor of the electric and magnetic fields as an oscillatory function, which, as convenience, we'll use a complex exponential;

$$
\mathbf{F}(\mathbf{r}, t) = \mathrm{Re}\{\mathbf{\tilde{F}}(\mathbf{r})\mathrm{e}^{\mathrm{i}\omega t}\}
$$

where $\mathbf{F}$ can be any of $\mathbf{E}, \mathbf{D}, \mathbf{B}, \mathbf{H}$, and $\omega = 2 \pi f$. Then we can apply the partial derivative with respect to $t$ in Ampère-Maxwell which results in the same exact field, but with a factor of $\mathrm{i}\omega$. If we replace $\mathbf{J}_f$ for $\sigma \mathbf{E}$ following Ohm's law, and if we then use this complex (or phasor) form, eliminate $\mathrm{Re}\{\}$ and $\mathrm{e}^{\mathrm{i}\omega t}$ from both sides of the equation, as they are not affected by derivatives with respect to space, we finally have

$$
\mathbf{\nabla} \times \mathbf{\tilde{H}} = \sigma\mathbf{\tilde{E}} + \varepsilon \mathrm{i} \omega \mathbf{\tilde{E}}
$$

Note that there is a term multiplied by a real factor, and another multiplied by an imaginary one. If we now consider Ampère-Maxwell as before, but with complex permittivity and no current density, we have that

$$
\mathbf{\nabla} \times \mathbf{\tilde{H}} = \varepsilon \mathrm{i} \omega \mathbf{\tilde{E}} = (\varepsilon' - \mathrm{i}\varepsilon'')\mathrm{i}\omega \mathbf{\tilde{E}}
$$

$$
\mathbf{\nabla} \times \mathbf{\tilde{H}} = \varepsilon''\omega\mathbf{\tilde{E}} + \varepsilon'\mathrm{i}\omega \mathbf{\tilde{E}}
$$

After this manipulation, you'll notice that we also have a real and an imaginary factor, and we can easily draw the correspondence $\sigma_{\mathrm{eff}} = \varepsilon'' \omega = \varepsilon_0 \varepsilon_{r}'' \omega$ from the real factor. Therefore, if we wish to model a material with a complex permittivity by using conductivity, one just needs $\varepsilon''(\omega)$, as well as $\omega$ itself, to calculate $\sigma_{\mathrm{eff}}$, which is then input into `electric_conductivity`.

Now, we should be careful with this. One must constantly adjust $\sigma_{\mathrm{eff}}$ as frequency is changed in order to obtain the correct $\varepsilon''$. Otherwise, if $\sigma_{\mathrm{eff}}$ is left constant as frequency is changed, the dependence on frequency would be quite different from what is observed in reality; $\varepsilon''(\omega)$ presents poles in many materials due to its nature, that is, it has peaks. By contrast, if this effective conductivity is not changed with frequency, the imaginary permittivity that FDTDX operates with varies as $1/\omega$, which has nothing to do with the physical value.

Currently, dispersion is not implemented in FDTDX. If more than a single incident wave traveling through the simulation volume is used, with different frequencies, then the simulation will be inaccurate, as the permittivity, and the conductivity, will both be the same for each incident wave. So, this means that you can only use one incident monochromatic wave. There's many situations where conductivity doesn't vary with frequency much; but it will be necessary to implement a frequency-dependent conductivity if imaginary permittivity is to be simulated using conductivity, as the former does vary a lot in many materials.

This is only just for describing dielectric media with no actual conductivity to speak of. If we consider both conductivity and complex permittivity, we have that

$$
\mathbf{\nabla} \times \mathbf{\tilde{H}} = (\sigma + \varepsilon''\omega)\mathbf{\tilde{E}} + \varepsilon' \mathrm{i} \omega \mathbf{\tilde{E}}
$$

then, in FDTDX, we'd have to set conductivity to be $\sigma_{\mathrm{eff}} = \sigma + \varepsilon''\omega$. Again, we need to know the value of $\sigma(\omega)$, $\varepsilon''(\omega)$ and $\omega$.

In this example notebook, we will demonstrate how to use such a lossy material, by propagating a plane wave through such a material, and we will compare the power left in the wave with an analytical result. We begin by importing the relevant packages,

In [12]:
import time

import jax
import jax.numpy as jnp
import pytreeclass as tc
from loguru import logger

import fdtdx

We define a function which analytically obtains the value of the power flowing
per unit area through the detector we will place in the simulation volume.

In [13]:
def irradiance(
    E0_0: float,
    freq: float,
    z: float,
    sigma: float,
    eps_r: float = 1.0,
    mu_r: float = 1.0,
) -> jax.Array:
    """
    Analytical calculation of time-averaged power-flux density ⟨S⟩·k̂ of an
    attenuating plane wave propagating in a lossy medium defined by real
    permittivity, real permeability, and real conductivity.

    Parameters
    ----------
    E0_0 : float
        Complex electric-field amplitude at z = 0 (peak value, V/m).
    freq : float
        Frequency of plane wave.
    z : float
        Distance from the reference plane (m), measured **along k̂**.
    sigma : float
        Conductivity as used in the vector form of Ohm's law. [S/m].
    eps_r : float, optional
        Relative permittivity ε_r (default 1.0).
    mu_r : float, optional
        Relative permeability μ_r (default 1.0).

    Returns
    -------
    I : jax.Array
        Irradiance in W m⁻².
    """
    omega = 2.0 * jnp.pi * freq
    eps = fdtdx.constants.eps0 * eps_r
    mu = fdtdx.constants.mu0 * mu_r
    # eta = jnp.sqrt(mu / eps)
    eta = jnp.sqrt((1j * omega * mu) / (sigma + (1j * omega * eps)))
    ratio = sigma / (omega * eps)
    root = jnp.sqrt(1.0 + ratio**2)
    alpha = jnp.sqrt((omega * mu * eps) / 2.0) * jnp.sqrt(root - 1.0)
    E0_z = E0_0 * jnp.exp(-alpha * z)  # field amplitude vs. z
    power_factor = jnp.real(eta) / jnp.abs(eta)
    return (jnp.square(jnp.abs(E0_z)) * power_factor) / (2.0 * jnp.abs(eta))

In [14]:
# Field/wave properties
wavelength = 7.4e-3
period = fdtdx.constants.wavelength_to_period(wavelength)
frequency = 1 / period
field_initial_amp = 1.0
# Material properties
# Relative permittivity (real)
eps_r = 2.0
# Relative permeability (real)
mu_r = 1.0
# Conductivity (real)
sigma = 5.0

Start the logger, set random seed and simulation configuration with duration in
time, `time`, length of each cube-shaped cell `resolution`, and the Courant
factor `courant_factor`,

In [15]:
exp_logger = fdtdx.Logger(
    experiment_name="simulate_lossy_material",
)
key = jax.random.PRNGKey(seed=42)
key, subkey = jax.random.split(key)

config = fdtdx.SimulationConfig(
    time=100e-11,
    resolution=1e-4,
    courant_factor=0.99,
)

30.06.2025 18:00:02.701 | /home/agalante/Repos/fdtdx-galcerte/src/fdtdx/utils/logger.py:124 - Starting experiment 
simulate_lossy_material in 
/home/agalante/Repos/fdtdx-galcerte/notebooks/outputs/nobackup/2025-06-30/simulate_lossy_material/18-00-02.677688

Logging field and material parameters used in the simulation.

In [16]:
period_steps = round(period / config.time_step_duration)
all_time_steps = list(range(config.time_steps_total))
logger.info(f"{config.time_steps_total=}")
logger.info(f"{period_steps=}")
logger.info(f"{config.max_travel_distance=}")
logger.info(f"{wavelength=}")
logger.info(f"{eps_r=}")
logger.info(f"{mu_r=}")

30.06.2025 18:00:02.810 | /tmp/ipykernel_486/296673337.py:3 - config.time_steps_total=5245

30.06.2025 18:00:02.814 | /tmp/ipykernel_486/296673337.py:4 - period_steps=129

30.06.2025 18:00:02.818 | /tmp/ipykernel_486/296673337.py:5 - config.max_travel_distance=0.29979245800000004

30.06.2025 18:00:02.820 | /tmp/ipykernel_486/296673337.py:6 - wavelength=0.0074

30.06.2025 18:00:02.827 | /tmp/ipykernel_486/296673337.py:7 - eps_r=2.0

30.06.2025 18:00:02.831 | /tmp/ipykernel_486/296673337.py:8 - mu_r=1.0

Initializing list where position constraints will be stored, as well as
instantiating the object holding information about the simulation volume.
We specify the dimensions in each axis in meters in `partial_real_shape`,
and we specify a background material permeating all of the volume in `material`.

In [ ]:
constraints = []

volume = fdtdx.SimulationVolume(
    partial_real_shape=(12e-3, 12e-3, 32e-3),
    material=fdtdx.Material(  # Background material
        permittivity=eps_r,
        permeability=mu_r,
        electric_conductivity=sigma,
    ),
)

Setting boundary conditions. We want periodic boundary conditions at the minimum
and maximum values of both X and Y axis, and a perfectly matched layer (PML)
simulating an open boundary on the minimum and maximum value of the Z axis.
It is also necessary to specfiy a certain thickness of the PML.

In [19]:
thickness = 10  # In grid units
kappa_start = 1.0
kappa_end = 1.5
bound_cfg = fdtdx.BoundaryConfig(
    boundary_type_minx="periodic",
    boundary_type_maxx="periodic",
    boundary_type_miny="periodic",
    boundary_type_maxy="periodic",
    boundary_type_minz="pml",
    boundary_type_maxz="pml",
    thickness_grid_minx=thickness,
    thickness_grid_maxx=thickness,
    thickness_grid_miny=thickness,
    thickness_grid_maxy=thickness,
    thickness_grid_minz=thickness,
    thickness_grid_maxz=thickness,
    kappa_start_minx=kappa_start,
    kappa_end_minx=kappa_end,
    kappa_start_maxx=kappa_start,
    kappa_end_maxx=kappa_end,
    kappa_start_miny=kappa_start,
    kappa_end_miny=kappa_end,
    kappa_start_maxy=kappa_start,
    kappa_end_maxy=kappa_end,
    kappa_start_minz=kappa_start,
    kappa_end_minz=kappa_end,
    kappa_start_maxz=kappa_start,
    kappa_end_maxz=kappa_end,
)
_, c_list = fdtdx.boundary_objects_from_config(bound_cfg, volume)
constraints.extend(c_list)

Setting a source for the field. We define an amplitude for the wave in
`amplitude`, and the wavelength is defined through a `WaveCharacter` object
passed to the parameter `wave_character`. We also set the polarization of the
field as a tuple holding the components (x, y, z) in
`fixed_E_polarization_vector`, and the propagation direction in `direction`.

After that, we extend the `constraints` list with the position this source
has in the simulation volume.

In [20]:
source = fdtdx.UniformPlaneSource(
    amplitude=field_initial_amp,
    partial_grid_shape=(None, None, 1),
    partial_real_shape=(None, None, None),
    fixed_E_polarization_vector=(1, 0, 0),
    # partial_grid_shape=(1, None, None),
    # partial_real_shape=(None, 10e-6, 10e-6),
    # fixed_E_polarization_vector=(0, 1, 0),
    # partial_grid_shape=(None, 1, None),
    # partial_real_shape=(10e-6, None, 10e-6),
    # fixed_E_polarization_vector=(1, 0, 0),
    wave_character=fdtdx.WaveCharacter(wavelength=wavelength),
    direction="-",
)
constraints.extend(
    [
        source.place_relative_to(
            volume,
            axes=(0, 1, 2),
            own_positions=(0, 0, 0),
            other_positions=(0, 0, 1),
            grid_margins=(0, 0, -(bound_cfg.thickness_grid_maxz + 4)),
        ),
    ]
)

We create the detector object. `PoyntingFluxDetector` measures the power flux per unit area
through its surface. If `reduce_volume` is set to `True`, then the power flux is summed
so that the total power flux through the detector is obtained.

In [21]:
detector = fdtdx.PoyntingFluxDetector(
    color=fdtdx.colors.LIGHT_GREEN,
    name="Detector",
    # This argument is for integrating the measured power flux over the surface
    reduce_volume=True,
    # An XY plane
    partial_grid_shape=(None, None, 1),
    direction="-",
    # When does the detector record data?
    switch=fdtdx.OnOffSwitch(fixed_on_time_steps=all_time_steps[:]),
)
constraints.append(
    detector.place_relative_to(
        source,
        axes=(2,),
        own_positions=(0,),
        other_positions=(0,),
        grid_margins=(-300,),
    )
)

After all the objects have been set and their positions determined, they are
placed into the simulation volume.

In [22]:
objects, arrays, params, config, _ = fdtdx.place_objects(
    volume=volume,
    config=config,
    constraints=constraints,
    key=subkey,
)
logger.info(tc.tree_summary(arrays, depth=1))
print(tc.tree_diagram(config, depth=4))

exp_logger.savefig(
    exp_logger.cwd,
    "setup.png",
    fdtdx.plot_setup(
        config=config,
        objects=objects,
    ),
)

2025-06-30 18:00:13.092359: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 20.5KiB (rounded to 20992)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-06-30 18:00:13.092474: W external/xla/xla/tsl/framework/bfc_allocator.cc:512] ***************************************************************************************************x
2025-06-30 18:00:23.093050: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 20.5KiB (rounded to 20992)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-06-30 18:00:23.093173: W exte

XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 20980 bytes.

We make a function that will hold the forward simulation as well as the
reverse-mode differentiation, in order to JIT compile it.

In [ ]:
def sim_fn(
    params: fdtdx.ParameterContainer,
    arrays: fdtdx.ArrayContainer,
    key: jax.Array,
):
    arrays, new_objects, info = fdtdx.apply_params(arrays, objects, params, key)

    final_state = fdtdx.run_fdtd(
        arrays=arrays,
        objects=new_objects,
        config=config,
        key=key,
    )
    _, arrays = final_state

    new_info = {
        **info,
    }
    return arrays, new_info

Finally, we compile the previously defined function, and execute it.

In [ ]:
compile_start_time = time.time()
jit_task_id = exp_logger.progress.add_task("JIT", total=None)
jitted_loss = jax.jit(sim_fn, donate_argnames=["arrays"]).lower(params, arrays, key).compile()
compile_delta_time = time.time() - compile_start_time
exp_logger.progress.update(jit_task_id, total=1, completed=1, refresh=True)

run_start_time = time.time()
key, subkey = jax.random.split(key)
arrays, info = jitted_loss(params, arrays, subkey)

runtime_delta = time.time() - run_start_time
info["runtime"] = runtime_delta
info["compile time"] = compile_delta_time

logger.info(f"{compile_delta_time=}")
logger.info(f"{runtime_delta=}")

# videos
exp_logger.log_detectors(iter_idx=0, objects=objects, detector_states=arrays.detector_states)